<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<h1 style="background:linear-gradient(135deg,#22313d 0%,#284740 100%);color:#edf5f3;padding:18px 22px;border-radius:20px;border:1px solid #3a5255;border-left:10px solid #78b0a1;box-shadow:0 10px 24px rgba(0,0,0,0.20);margin:0 0 18px 0;">GRPO (Group Relative Policy Optimization) - Complete Interview Guide</h1>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Table of Contents</h2>
<ol style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#The-Foundation-of-GRPO">The Foundation of GRPO</a> - Policy Model, Reference Model, Reward Model, Value Model</li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#What-is-GRPO?">What is GRPO?</a> - Definition and core concept</li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#Why-GRPO?">Why GRPO?</a> - Challenges with PPO and how GRPO solves them</li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#Key-Idea">Key Idea</a> - Relative evaluation and group-based advantages</li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#LLM-as-a-Policy">LLM as a Policy</a> - Policy network and token generation</li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#Reward-and-Advantage-Calculation">Reward and Advantage Calculation</a> - Per-token rewards and baseline advantages</li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#Understanding-the-GRPO-Objective-Function">The GRPO Objective Function</a> - Full mathematical breakdown</li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#Step-by-Step-Breakdown">Step-by-Step Example</a> - Walkthrough with math (query, responses, rewards, advantage, clipping, KL divergence)</li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#GRPO-vs-PPO-vs-DPO-Comparison">GRPO vs PPO vs DPO Comparison</a> - Head-to-head comparison table</li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#Advantages-and-Limitations-of-GRPO">Advantages and Limitations of GRPO</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#DeepSeek-R1:-How-GRPO-Enabled-Reasoning-Without-Supervised-Data">DeepSeek-R1: How GRPO Enabled Reasoning Without Supervised Data</a></li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#When-to-Use-GRPO-vs-Other-Methods">When to Use GRPO vs Other Methods</a> - Decision guide</li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#Top-8-GRPO-Interview-Questions-and-Answers">Top 8 GRPO Interview Q&amp;A</a> - 30-second answers</li>
<li style="margin:6px 0;"><a style="color:#8fd1ff;" href="#References">References</a></li>
</ol>
</div>


# The Foundation of GRPO

1.  Policy Model - Fancy name for the current `LLM you are training`
2.  Reference Model - A frozen version of the original LLM you are training
3.  Reward Model - The model that was trained on human preferences (from the technique in InstructGPT above)
4.  Value Model - `A model that is trying to estimate the long term reward given certain actions`

![Why GRPO is Important and How it Works](https://ghost.oxen.ai/content/images/size/w960/2025/02/Screenshot-2025-02-11-at-7.58.22-PM.png)

## What is GRPO?

Group Relative Policy Optimization (GRPO) is a reinforcement learning (RL) algorithm specifically designed to enhance `reasoning capabilities in Large Language Models `(LLMs). 

## Why GRPO?

Traditional RL methods like Proximal Policy Optimization (PPO) face significant challenges when applied to reasoning tasks in LLMs:

**Dependency on a Critic Model**:
-   PPO requires a separate critic model to estimate the value of each response, which doubles memory and computational requirements.
-   `Training the critic` is complex and prone to errors, especially for tasks with subjective or nuanced evaluations.

**High Computational Cost**:
-   RL pipelines often demand significant computational resources to evaluate and optimize responses iteratively.
-   Scaling these methods to large LLMs exacerbates these costs.

**Scalability Issues**:
-   Absolute reward evaluations struggle with diverse tasks, making it hard to generalize across reasoning domains.

**How GRPO Addresses These Challenges**:
-   **Critic-Free Optimization**:  GRPO removes the need for a critic model  by comparing responses within a group, significantly reducing computational overhead.
-   **Relative Evaluation**: Instead of relying on an external evaluator, `GRPO uses group dynamics to assess how well a response performs relative to others in the same batch`.
-   **Efficient Training**: By focusing on `group-based advantages`, GRPO simplifies the reward estimation process,  making it faster and more scalable for large models.

## Key Idea

At the heart of GRPO is the concept of  **relative evaluation**:

-   For each input query, the model generates a group of potential responses.
-   These responses are scored based on how they compare to others in the group, rather than being evaluated in isolation.
-   The  **advantage**  of a response reflects how much better or worse it is relative to the group’s average performance.

This approach eliminates the need for a separate critic,  making GRPO both efficient and robust. 


## LLM as a Policy

In GRPO, the language model serves as the policy network (actor), taking a question  **_q_**  as input observation  **_s_**  and producing a sequence of tokens as actions. The policy distribution factors across tokens:

![](https://archive.is/IIEHh/1c74361d416d64991eb38e95d1bd81ba8bbe0a98.webp)

# Sequential Token Generation

The generation process is inherently sequential because of auto-regressive nature of transformers/LLMs:

1.  Each token is generated conditionally on previous tokens
2.  The policy network (LLM) maintains a running context
3.  Each token generation step can be viewed as an action  _a_t_ in the RL framework

![](https://archive.is/IIEHh/660ee7ee8470676857bf3119453f303657b07d9c.webp)

# Reward and Advantage Calculation

For each generated sequence, GRPO computes per-token rewards as follows:

![](https://archive.is/IIEHh/538dbcb78755322addc793d48444df2ebb176e41.webp)

Instead of using a ~~value network~~, GRPO estimates baseline advantages  **_A_**  by normalizing a group (batch) of  **rewards**  obtained from  **sampling multiple different outputs**  from the  **reference policy**  produced in response to the same question as input:

![](https://archive.is/IIEHh/ac9a38de5f9392e0b3fcceab358a95eb2544331e.webp)

The GRPO Objective

for each question 𝑞, GRPO samples a group of outputs {𝑜1, 𝑜2, · · · , 𝑜𝐺} from the old policy 𝜋𝜃𝑜𝑙𝑑 and then optimizes the policy model by maximizing the GRPO objective. The complete GRPO objective brings everything together:

![](https://archive.is/IIEHh/6b7a7fba5780c07b36518580a0def98f26fa5083.webp)

This objective:

1.  Averages over both groups and sequence lengths
2.  Uses clipping for conservative updates
3.  Includes an  **estimate**  of the KL divergence as a penalty to prevent large deviations from the reference model

![](https://archive.is/IIEHh/da646d6d1e6db20ca38d292ff6c9d5ab77368f2e.webp)

# Understanding the GRPO Objective Function

The objective function in Group Relative Policy Optimization (GRPO) defines how the model learns to improve its policy, driving its ability to generate high-quality responses. Let’s break it down step by step.

## The GRPO Objective Function

![](https://archive.is/oTSHb/4585fd2b687ecaec85de580c3bdb9cec76d27e0f.webp)

![](https://archive.is/oTSHb/8fb4f356b071fdf0bca698d3938667978d4ec33e.webp)

![](https://archive.is/oTSHb/4188beef80e4e8fb99e187072163c9c6b499b025.webp)


#### Example

# Step-by-Step Breakdown

## Step 1: Start with a Query

-   Pick a query (q) from the training dataset P(Q)  
    _Example_: Let’s say the query is  **“What is the sum of 8 + 5?”**

## Step 2: Generate a Group of Responses

-   The model generates a group of GGG responses to the query.  
    _Example_: The model generates these responses:
-   o1​: “The answer is 13.”
-   o2​: “Thirteen.”
-   o3​: “It’s 12.”
-   o4: “The sum is 13.”

## Step 3: Calculate Rewards for Each Response

**What are Rewards?**:

-   Rewards guide the model’s learning by quantifying the quality of its responses.

**Types of Rewards in GRPO**:

-   **Accuracy Rewards**: Based on the correctness of the response (e.g., solving a math problem).
-   **Format Rewards**: Ensures the response adheres to structural guidelines (e.g., reasoning enclosed in  `<think>`  tags).
-   **Language Consistency Rewards**: Penalizes language mixing or incoherent formatting.

Assign a  **reward (ri)**  to each response based on how good it is. For example Rewards could depend on:

**Accuracy**: Is the answer correct?

**Format**: Is the response well-structured?  
_Example_:

-   r1=1.0 (correct and well-formatted).
-   r2=0.9 (correct but less formal).
-   r3=0.0 (incorrect answer).
-   r4=1.0 (correct and well-formatted).

## Step 4: Compare Responses (Group Advantage)

-   Calculate the  **advantage (Ai​)**  of each response relative to the group:

![](https://archive.is/oTSHb/3204fd7be2544b9ea8ca5524366b1021034bc934.webp)

Equations are from the paper and explanation was created with the help from GPT-4o

In simple way you can understand it like this

![](https://archive.is/oTSHb/6d4a12ffae8e696e8c60c4845b0c14698f04d5b1.webp)

Equations are from the paper and explanation was created with the help from GPT-4o

-   Responses better than the group’s average get positive scores, while worse responses get negative scores.
-   Encourages competition within the group, driving the model to generate better responses.

## Step 5: Update Policy with Clipping


![](https://archive.is/oTSHb/91b12ebf0ffa94274f2b8d3129a1dfddcd8d848d.webp)

![](https://ghost.oxen.ai/content/images/2025/02/9.png)

> The main signal you are trying to get out of your LLMs during RL is represented by “A” which stands for the “Advantage”. 

This helps give `direction to update the original LLM’s weights`. If the the advantage is high, you want to encourage the model to keep doing the same actions. If it is low, you want to encourage the model to try something different.

-   _Example_: If the new policy starts assigning too much probability to o1, clipping ensures it doesn’t overemphasize this response.
-   Enables steady and reliable policy optimization, even in complex tasks like reasoning.

## Step 6: Penalize Deviations with KL Divergence

![](https://archive.is/oTSHb/b80a90dbc9f0a37f188560f4e901d7d25df54d2c.webp)

The idea is that we do not want to drift too far from the original model. For each token, we want to make sure the new predictions do not drift too far from the original ones.
![](https://ghost.oxen.ai/content/images/2025/02/14.png)
The intuition behind enforcing the KL Divergence is that the model we are starting with already knows how to write coherent sentences and follow instructions. `We don’t want the new model to “reward hack” or exploit some sort of property in our reward signal that is not aligned with the original model.`

> ## The Reward Signals

What’s super interesting about the DeepSeek-R1-Zero work is that they go even further to slash the memory usage because don’t use a “neural reward model”.


![](https://ghost.oxen.ai/content/images/2025/02/2.png)

<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<hr style="border:0;height:1px;background:linear-gradient(90deg,transparent,#556977,transparent);margin:20px 0;" />
<h1 style="background:linear-gradient(135deg,#22313d 0%,#284740 100%);color:#edf5f3;padding:18px 22px;border-radius:20px;border:1px solid #3a5255;border-left:10px solid #78b0a1;box-shadow:0 10px 24px rgba(0,0,0,0.20);margin:0 0 18px 0;">GRPO vs PPO vs DPO Comparison</h1>
<table style="width:100%;border-collapse:collapse;background:#ffffff;font-size:0.96rem;margin:16px 0;border:1px solid #dde6ec;border-radius:16px;overflow:hidden;">
<thead>
<tr>
<th style="background:#f3f6f8;color:#24415c;padding:10px 12px;text-align:left;border:1px solid #dde6ec;font-weight:700;">Feature</th>
<th style="background:#f3f6f8;color:#24415c;padding:10px 12px;text-align:left;border:1px solid #dde6ec;font-weight:700;"><strong>GRPO</strong></th>
<th style="background:#f3f6f8;color:#24415c;padding:10px 12px;text-align:left;border:1px solid #dde6ec;font-weight:700;"><strong>PPO</strong></th>
<th style="background:#f3f6f8;color:#24415c;padding:10px 12px;text-align:left;border:1px solid #dde6ec;font-weight:700;"><strong>DPO</strong></th>
</tr>
</thead>
<tbody>
<tr style="background:#f8fbff;">
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>Needs Reward Model?</strong></td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Yes (or rule-based rewards)</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Yes</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">No (implicit in preference pairs)</td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>Needs Critic / Value Model?</strong></td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">No -- uses group-relative baseline</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Yes -- separate value network required</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">No</td>
</tr>
<tr style="background:#f8fbff;">
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>Number of Models in Memory</strong></td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">2 (policy + reference)</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">4 (policy + reference + reward + critic)</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">2 (policy + reference)</td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>Compute Cost</strong></td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Medium -- no critic, but samples G outputs per query</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">High -- critic training + per-token value estimation</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Low -- single supervised-style pass</td>
</tr>
<tr style="background:#f8fbff;">
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>Training Stability</strong></td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">High -- clipping + group normalization smooths updates</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Medium -- sensitive to reward model quality and critic errors</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">High -- no RL loop, purely supervised</td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>Data Efficiency</strong></td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Medium -- needs multiple sampled outputs per query</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Low -- requires large amounts of on-policy data</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">High -- directly uses preference pairs</td>
</tr>
<tr style="background:#f8fbff;">
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>Handles Reasoning Tasks</strong></td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Excellent -- designed for multi-step reasoning with verifiable rewards</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Good -- but critic struggles with long-horizon reasoning</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Limited -- preference pairs hard to collect for reasoning</td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>Key Innovation</strong></td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Replaces critic with group-relative advantage estimation</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Actor-critic with clipped surrogate objective</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Reparameterizes reward into policy directly</td>
</tr>
<tr style="background:#f8fbff;">
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>Risk of Reward Hacking</strong></td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Lower -- KL penalty + relative scoring reduces exploitation</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Higher -- absolute rewards easier to exploit</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Lowest -- no explicit reward signal</td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>Scalability to Large LLMs</strong></td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Excellent -- ~50% memory savings over PPO</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Poor -- 4 models in memory is prohibitive</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Good -- lightweight training loop</td>
</tr>
<tr style="background:#f8fbff;">
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>Used By</strong></td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">DeepSeek-R1, DeepSeek-R1-Zero</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">InstructGPT, early ChatGPT</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Llama-2-Chat, Zephyr, many open-source models</td>
</tr>
</tbody>
</table>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Key Takeaway</strong>: GRPO sits in a sweet spot -- it retains the flexibility of online RL (like PPO) to explore and improve through sampling, while slashing compute costs by eliminating the critic model. DPO is simpler but cannot explore beyond its fixed preference dataset.</p>
</div>


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<hr style="border:0;height:1px;background:linear-gradient(90deg,transparent,#556977,transparent);margin:20px 0;" />
<h1 style="background:linear-gradient(135deg,#22313d 0%,#284740 100%);color:#edf5f3;padding:18px 22px;border-radius:20px;border:1px solid #3a5255;border-left:10px solid #78b0a1;box-shadow:0 10px 24px rgba(0,0,0,0.20);margin:0 0 18px 0;">Advantages and Limitations of GRPO</h1>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Advantages</h2>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>No Critic Model Needed</strong>: Eliminates the value network entirely, saving ~50% GPU memory compared to PPO. This is the single biggest practical benefit for training large LLMs.</li>
<li style="margin:6px 0;"><strong>Simpler Training Pipeline</strong>: Only 2 models in memory (policy + reference) instead of PPO's 4 models. Fewer moving parts means fewer failure modes.</li>
<li style="margin:6px 0;"><strong>Robust Advantage Estimation</strong>: Group-relative normalization (z-score across sampled outputs) provides a stable, self-calibrating baseline that adapts to task difficulty automatically.</li>
<li style="margin:6px 0;"><strong>Works with Rule-Based Rewards</strong>: Does not strictly require a learned reward model. DeepSeek-R1-Zero used simple correctness checks (e.g., &quot;is the final answer right?&quot;) and format rules, avoiding reward model training entirely.</li>
<li style="margin:6px 0;"><strong>Naturally Suited for Reasoning</strong>: By sampling multiple complete chain-of-thought outputs and comparing them, GRPO can distinguish between correct and incorrect reasoning paths -- something a per-token critic struggles with.</li>
<li style="margin:6px 0;"><strong>Exploration Through Sampling</strong>: Unlike DPO (which is offline), GRPO generates new outputs each iteration, allowing the model to discover novel reasoning strategies over time.</li>
<li style="margin:6px 0;"><strong>Scalable</strong>: The memory and compute savings make it feasible to apply RL to models with 100B+ parameters where PPO would be impractical.</li>
</ul>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Limitations</h2>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>Sampling Overhead</strong>: Generating G outputs per query (typically G=8 to 64) adds inference cost. Each training step requires multiple forward passes through the policy.</li>
<li style="margin:6px 0;"><strong>Reward Design is Critical</strong>: The quality of learning depends heavily on how rewards are defined. For open-ended tasks (creative writing, summarization), designing good rule-based rewards is non-trivial.</li>
<li style="margin:6px 0;"><strong>Group Size Sensitivity</strong>: Too small a group (G &lt; 4) gives noisy advantage estimates; too large a group increases compute cost. Requires tuning.</li>
<li style="margin:6px 0;"><strong>No Per-Token Credit Assignment</strong>: GRPO assigns the same advantage to all tokens in a response. It cannot pinpoint which specific token or reasoning step was responsible for success/failure (unlike PPO's per-token value estimates).</li>
<li style="margin:6px 0;"><strong>Limited to Tasks with Verifiable Outcomes</strong>: Works best when you can objectively score outputs (math, code, factual QA). Less clear how to apply it to subjective tasks.</li>
<li style="margin:6px 0;"><strong>Still Requires a Reference Model</strong>: The KL divergence penalty requires keeping the reference model in memory, so you still need 2x the model parameters (though PPO needs 4x).</li>
</ul>
</div>


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<hr style="border:0;height:1px;background:linear-gradient(90deg,transparent,#556977,transparent);margin:20px 0;" />
<h1 style="background:linear-gradient(135deg,#22313d 0%,#284740 100%);color:#edf5f3;padding:18px 22px;border-radius:20px;border:1px solid #3a5255;border-left:10px solid #78b0a1;box-shadow:0 10px 24px rgba(0,0,0,0.20);margin:0 0 18px 0;">DeepSeek-R1: How GRPO Enabled Reasoning Without Supervised Data</h1>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">The Breakthrough</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">DeepSeek-R1 (January 2025) demonstrated that <strong>pure reinforcement learning with GRPO can teach an LLM to reason</strong> -- without any supervised fine-tuning data, human-written chain-of-thought examples, or a learned neural reward model. This was a landmark result.</p>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">DeepSeek-R1-Zero: The Pure RL Experiment</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">DeepSeek-R1-Zero was trained using <strong>only GRPO</strong> starting from the DeepSeek-V3 base model:</p>
<ol style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>No SFT Stage</strong>: Skipped supervised fine-tuning entirely. The base model went straight into RL.</li>
<li style="margin:6px 0;"><strong>Rule-Based Rewards Only</strong>: Used two simple reward signals:
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>Accuracy Reward</strong>: Binary -- did the model get the correct final answer? (for math/code tasks)</li>
<li style="margin:6px 0;"><strong>Format Reward</strong>: Did the model place its reasoning inside <code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">&lt;think&gt;</code> tags and its answer inside <code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">&lt;answer&gt;</code> tags?</li>
</ul>
</li>
<li style="margin:6px 0;"><strong>No Neural Reward Model</strong>: No human preference data was collected, no reward model was trained. This eliminated an entire stage of the RLHF pipeline.</li>
</ol>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Why GRPO Was Essential</h2>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>Memory Efficiency</strong>: Training a 671B parameter model (DeepSeek-V3) with PPO would require 4 copies of the model in memory -- completely infeasible. GRPO's critic-free design made RL at this scale possible.</li>
<li style="margin:6px 0;"><strong>Group Sampling Discovers Reasoning</strong>: By generating multiple outputs per query and comparing them, the model naturally discovered that longer, step-by-step reasoning leads to higher accuracy rewards. This <strong>emergent chain-of-thought</strong> appeared without being explicitly taught.</li>
<li style="margin:6px 0;"><strong>Self-Verification Emerged</strong>: The model spontaneously learned to re-check its work, allocate more &quot;thinking tokens&quot; to harder problems, and try alternative approaches when stuck -- all from the RL signal alone.</li>
</ul>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">The &quot;Aha Moment&quot;</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">During training, DeepSeek researchers observed the model's outputs evolving:</p>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;"><strong>Early training</strong>: Short, direct answers (often wrong)</li>
<li style="margin:6px 0;"><strong>Mid training</strong>: Model starts producing longer responses with rudimentary reasoning steps</li>
<li style="margin:6px 0;"><strong>Late training</strong>: Sophisticated multi-step reasoning with self-correction, backtracking, and verification</li>
</ul>
<blockquote style="margin:14px 0;padding:12px 16px;background:#24313d;border-left:5px solid #90b7a2;border-radius:14px;border:1px solid #3c4d5a;color:#d9e4ec;">
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">This progression happened purely from GRPO's reward signal -- the model learned that &quot;thinking more carefully&quot; = higher group-relative advantage.</p>
</blockquote>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">DeepSeek-R1 (Full Pipeline)</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">The final DeepSeek-R1 model added a small amount of SFT data to fix readability issues from R1-Zero (language mixing, poor formatting), then continued with GRPO. The pipeline was:</p>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">Base Model</code> --&gt; <code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">Cold Start SFT (small amount)</code> --&gt; <code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">GRPO RL Training</code> --&gt; <code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">Rejection Sampling + SFT</code> --&gt; <code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">Final GRPO RL</code></p>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Key Insight for Interviews</h2>
<blockquote style="margin:14px 0;padding:12px 16px;background:#24313d;border-left:5px solid #90b7a2;border-radius:14px;border:1px solid #3c4d5a;color:#d9e4ec;">
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;">GRPO made it possible to apply RL to a 671B parameter model by eliminating the critic. The group-relative advantage was sufficient to discover emergent reasoning capabilities -- proving that <strong>RL alone can teach LLMs to think step-by-step</strong>, without human-written chain-of-thought demonstrations.</p>
</blockquote>
</div>


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<hr style="border:0;height:1px;background:linear-gradient(90deg,transparent,#556977,transparent);margin:20px 0;" />
<h1 style="background:linear-gradient(135deg,#22313d 0%,#284740 100%);color:#edf5f3;padding:18px 22px;border-radius:20px;border:1px solid #3a5255;border-left:10px solid #78b0a1;box-shadow:0 10px 24px rgba(0,0,0,0.20);margin:0 0 18px 0;">When to Use GRPO vs Other Methods</h1>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Decision Guide</h2>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Use GRPO when:</strong></p>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;">You have tasks with <strong>verifiable/objective outcomes</strong> (math, code, factual QA, structured output generation)</li>
<li style="margin:6px 0;">You want the model to <strong>explore and discover</strong> new reasoning strategies (not just mimic existing data)</li>
<li style="margin:6px 0;">You are training a <strong>very large model</strong> (50B+ parameters) where PPO's 4-model memory footprint is infeasible</li>
<li style="margin:6px 0;">You can define <strong>simple rule-based rewards</strong> (correctness checks, format validation) without needing a neural reward model</li>
<li style="margin:6px 0;">You want <strong>emergent chain-of-thought reasoning</strong> without collecting human-written reasoning traces</li>
</ul>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Use PPO when:</strong></p>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;">You have a <strong>high-quality neural reward model</strong> trained on human preferences</li>
<li style="margin:6px 0;">Your task requires <strong>fine-grained, per-token credit assignment</strong> (e.g., the model needs to learn that a specific word choice was bad)</li>
<li style="margin:6px 0;">You have <strong>sufficient compute budget</strong> for 4 models in memory</li>
<li style="margin:6px 0;">You are working with <strong>smaller models</strong> (&lt; 13B) where the memory overhead is manageable</li>
</ul>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Use DPO when:</strong></p>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;">You have a <strong>high-quality preference dataset</strong> (human-annotated pairs of &quot;chosen&quot; vs &quot;rejected&quot; responses)</li>
<li style="margin:6px 0;">Your task is <strong>subjective</strong> (creative writing, summarization, helpfulness) where rule-based rewards are hard to define</li>
<li style="margin:6px 0;">You want the <strong>simplest training pipeline</strong> with no RL loop at all</li>
<li style="margin:6px 0;">You want a <strong>stable, predictable training process</strong> with supervised-style loss</li>
<li style="margin:6px 0;">You do <strong>not need the model to explore</strong> beyond behaviors present in your preference data</li>
</ul>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Use RLHF (PPO-based) when:</strong></p>
<ul style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;">You are building a <strong>general-purpose assistant</strong> where human preferences are the ground truth</li>
<li style="margin:6px 0;">You need <strong>online exploration</strong> but also have a strong reward model</li>
<li style="margin:6px 0;">You can afford the compute cost</li>
</ul>
<h2 style="background:linear-gradient(90deg,#22313d 0%,#273743 100%);border-left:8px solid #72aa9b;color:#e7eff3;padding:11px 16px;border-radius:14px;border:1px solid #3a4b58;box-shadow:0 6px 18px rgba(0,0,0,0.18);margin:22px 0 14px 0;">Quick Reference Matrix</h2>
<table style="width:100%;border-collapse:collapse;background:#ffffff;font-size:0.96rem;margin:16px 0;border:1px solid #dde6ec;border-radius:16px;overflow:hidden;">
<thead>
<tr>
<th style="background:#f3f6f8;color:#24415c;padding:10px 12px;text-align:left;border:1px solid #dde6ec;font-weight:700;">Scenario</th>
<th style="background:#f3f6f8;color:#24415c;padding:10px 12px;text-align:left;border:1px solid #dde6ec;font-weight:700;">Best Method</th>
</tr>
</thead>
<tbody>
<tr style="background:#f8fbff;">
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Math/Code reasoning for large LLM</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>GRPO</strong></td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">General chatbot alignment with preference data</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>DPO</strong></td>
</tr>
<tr style="background:#f8fbff;">
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Small model with good reward model</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>PPO</strong></td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Emergent reasoning without SFT data</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>GRPO</strong></td>
</tr>
<tr style="background:#f8fbff;">
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Creative writing improvement</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>DPO</strong></td>
</tr>
<tr>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Large-scale RLHF with budget</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>PPO</strong></td>
</tr>
<tr style="background:#f8fbff;">
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;">Structured output / format compliance</td>
<td style="padding:10px 12px;border:1px solid #e4edf2;vertical-align:top;color:#31475b;"><strong>GRPO</strong></td>
</tr>
</tbody>
</table>
</div>


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<hr style="border:0;height:1px;background:linear-gradient(90deg,transparent,#556977,transparent);margin:20px 0;" />
<h1 style="background:linear-gradient(135deg,#22313d 0%,#284740 100%);color:#edf5f3;padding:18px 22px;border-radius:20px;border:1px solid #3a5255;border-left:10px solid #78b0a1;box-shadow:0 10px 24px rgba(0,0,0,0.20);margin:0 0 18px 0;">Top 8 GRPO Interview Questions and Answers</h1>
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">Q1: What is GRPO and how does it differ from PPO?</h3>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Answer</strong>: GRPO (Group Relative Policy Optimization) is an RL algorithm for LLM training that eliminates the critic/value model used in PPO. Instead of estimating per-token values with a separate network, GRPO samples a group of G outputs for each query and computes advantages by normalizing rewards within the group (z-score). This cuts memory from 4 models (PPO) to 2 models (policy + reference), making RL feasible for very large LLMs like DeepSeek-R1's 671B parameter model.</p>
<hr style="border:0;height:1px;background:linear-gradient(90deg,transparent,#556977,transparent);margin:20px 0;" />
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">Q2: How does GRPO compute the advantage without a critic?</h3>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Answer</strong>: For each query, GRPO samples G outputs from the current policy and scores each with a reward function. The advantage of output i is: <code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">A_i = (r_i - mean(r)) / std(r)</code>, where mean and std are computed over the group. Outputs better than the group average get positive advantage (reinforced), worse ones get negative advantage (discouraged). This group-relative baseline replaces the learned value function in PPO.</p>
<hr style="border:0;height:1px;background:linear-gradient(90deg,transparent,#556977,transparent);margin:20px 0;" />
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">Q3: What is the role of KL divergence in GRPO?</h3>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Answer</strong>: The KL divergence penalty prevents the policy from drifting too far from the reference model (a frozen copy of the initial model). Without it, the model could &quot;reward hack&quot; -- finding degenerate outputs that score high on the reward function but are incoherent. The KL term is added to the objective as: <code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">J_GRPO = clipped_advantage - beta * KL(policy || reference)</code>. The coefficient beta controls how conservative updates are.</p>
<hr style="border:0;height:1px;background:linear-gradient(90deg,transparent,#556977,transparent);margin:20px 0;" />
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">Q4: Why was GRPO critical for DeepSeek-R1?</h3>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Answer</strong>: DeepSeek-R1 is based on DeepSeek-V3, a 671B parameter MoE model. PPO would require 4 copies of this model in memory (policy, reference, reward, critic) -- completely infeasible. GRPO requires only 2 (policy + reference). Additionally, GRPO enabled training with simple rule-based rewards (accuracy + format), eliminating the need for a neural reward model. The result: emergent chain-of-thought reasoning discovered purely through RL, without supervised reasoning examples.</p>
<hr style="border:0;height:1px;background:linear-gradient(90deg,transparent,#556977,transparent);margin:20px 0;" />
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">Q5: What are the reward signals used in DeepSeek-R1-Zero's GRPO training?</h3>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Answer</strong>: Two rule-based rewards, no neural reward model: (1) <strong>Accuracy reward</strong> -- binary signal checking if the final answer is correct (for math/code, using ground-truth verification); (2) <strong>Format reward</strong> -- checks if the response uses the required <code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">&lt;think&gt;...&lt;/think&gt;</code> and <code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">&lt;answer&gt;...&lt;/answer&gt;</code> structure. These simple signals were sufficient for the model to develop sophisticated multi-step reasoning.</p>
<hr style="border:0;height:1px;background:linear-gradient(90deg,transparent,#556977,transparent);margin:20px 0;" />
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">Q6: What does &quot;group&quot; mean in GRPO, and how does group size affect training?</h3>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Answer</strong>: The &quot;group&quot; is the set of G outputs sampled from the policy for a single query. Typical values are G=8 to 64. Larger groups give more stable advantage estimates (better statistical baseline) but cost more compute (G forward passes per query). Smaller groups are noisier but faster. The group size is a key hyperparameter -- too small (G &lt; 4) leads to high-variance updates, too large wastes compute with diminishing returns.</p>
<hr style="border:0;height:1px;background:linear-gradient(90deg,transparent,#556977,transparent);margin:20px 0;" />
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">Q7: How does GRPO compare to DPO for reasoning tasks?</h3>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Answer</strong>: GRPO is better suited for reasoning because: (1) it performs <strong>online exploration</strong> -- generating new outputs each iteration, allowing discovery of novel reasoning strategies; (2) it works with <strong>verifiable rewards</strong> (correct/incorrect) rather than requiring human preference pairs, which are expensive to collect for reasoning; (3) DPO is <strong>offline</strong> -- it only learns from its fixed preference dataset and cannot discover reasoning approaches not present in the data. However, DPO is simpler and better for subjective tasks like helpfulness or writing style.</p>
<hr style="border:0;height:1px;background:linear-gradient(90deg,transparent,#556977,transparent);margin:20px 0;" />
<h3 style="display:inline-block;color:#d8e4ea;background:#22303a;border-left:5px solid #7fa89e;padding:8px 14px;border-radius:12px;border:1px solid #3a4a52;margin:18px 0 10px 0;">Q8: What is the &quot;clipping&quot; mechanism in GRPO and why is it needed?</h3>
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Answer</strong>: GRPO uses the same clipping mechanism as PPO: the probability ratio <code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">r_t = pi_new(token) / pi_old(token)</code> is clipped to the range <code style="background:#27313a;color:#e8eef4;padding:2px 6px;border-radius:6px;border:1px solid #3c4a55;">[1-epsilon, 1+epsilon]</code> (typically epsilon=0.2). This prevents any single update from changing the policy too drastically. Without clipping, a response with very high advantage could cause the policy to collapse to always producing that specific output, destroying diversity. The clipped objective takes the minimum of the unclipped and clipped versions, ensuring conservative updates.</p>
</div>


<div data-notebook-cell-style-version="2026-04-v2" style="color:#d7e0e7;line-height:1.8;font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
<hr style="border:0;height:1px;background:linear-gradient(90deg,transparent,#556977,transparent);margin:20px 0;" />
<h1 style="background:linear-gradient(135deg,#22313d 0%,#284740 100%);color:#edf5f3;padding:18px 22px;border-radius:20px;border:1px solid #3a5255;border-left:10px solid #78b0a1;box-shadow:0 10px 24px rgba(0,0,0,0.20);margin:0 0 18px 0;">References</h1>
<ol style="margin:10px 0 16px 20px;padding-left:18px;line-height:1.9;color:#d2dce4;">
<li style="margin:6px 0;">
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>DeepSeek-R1 Paper</strong>: Shao, Z., et al. &quot;DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning.&quot; DeepSeek-AI, January 2025. <a style="color:#8fd1ff;" href="https://arxiv.org/abs/2501.12948">arXiv:2501.12948</a></p>
</li>
<li style="margin:6px 0;">
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>GRPO Original Paper (DeepSeekMath)</strong>: Shao, Z., et al. &quot;DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models.&quot; DeepSeek-AI, 2024. <a style="color:#8fd1ff;" href="https://arxiv.org/abs/2402.03300">arXiv:2402.03300</a></p>
</li>
<li style="margin:6px 0;">
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>PPO (Proximal Policy Optimization)</strong>: Schulman, J., et al. &quot;Proximal Policy Optimization Algorithms.&quot; OpenAI, 2017. <a style="color:#8fd1ff;" href="https://arxiv.org/abs/1707.06347">arXiv:1707.06347</a></p>
</li>
<li style="margin:6px 0;">
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>DPO (Direct Preference Optimization)</strong>: Rafailov, R., et al. &quot;Direct Preference Optimization: Your Language Model is Secretly a Reward Model.&quot; Stanford, 2023. <a style="color:#8fd1ff;" href="https://arxiv.org/abs/2305.18290">arXiv:2305.18290</a></p>
</li>
<li style="margin:6px 0;">
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>InstructGPT / RLHF</strong>: Ouyang, L., et al. &quot;Training language models to follow instructions with human feedback.&quot; OpenAI, 2022. <a style="color:#8fd1ff;" href="https://arxiv.org/abs/2203.02155">arXiv:2203.02155</a></p>
</li>
<li style="margin:6px 0;">
<p style="margin:10px 0 15px 0;line-height:1.82;color:#d7e0e7;font-size:1rem;"><strong>Oxen.ai Blog - GRPO Explained</strong>: &quot;Why GRPO is Important and How it Works.&quot; Oxen.ai, February 2025. <a style="color:#8fd1ff;" href="https://ghost.oxen.ai/content/images/size/w960/2025/02/Screenshot-2025-02-11-at-7.58.22-PM.png">https://ghost.oxen.ai</a></p>
</li>
</ol>
</div>
